# P1T validation run, amendment A1: development + validation periods only (TWSE)

Registered spec: `P1T_taiwan_spec.md`. This notebook:
- downloads **2013-01-02 to 2018-12-28** only;
- has a hard stop at 2018-12-28 — the holdout (2019 onward) is never requested;
- runs the registered executor once on the validation period.

Raw daily responses are saved to Google Drive (`MyDrive/P1T_raw/`, about 1 GB) with SHA-256, so the run can resume
after a disconnect. Downloading takes about 3 hours (≈ 3,000 requests at one every 3.5 s, the pace TWSE tolerates).
Return `p1t_validation_results.zip`.

**Run the single code cell once.** Rerunning it resumes the download and repeats nothing already stored.

**Amendment A1 (7 November 2026).** The first attempt stopped while parsing, before any statistic was computed: TWSE's T86 report uses three column schemas over 2013–2018. The executor now maps each schema to the registered classes (`P1T_taiwan_spec.md` §11). All raw files are already on Drive, so this run downloads nothing new and takes a few minutes. It also records flow-only accounting checks per schema era (`t86_diagnostics`).

In [ ]:

import os, io, json, gzip, time, zipfile, hashlib, datetime, re, sys
import numpy as np, requests
EXECUTOR_SHA256 = "da44075ed9da3a240a792f74777b50404b96401e90e1075e5433e6613e7d7fa0"
EXECUTOR_SRC = "\"\"\"P1T executor: Taiwan tagged-flow forecast-gain test (registered spec P1T_taiwan_spec.md).\n\nLayers\n------\n1. Core (pure numpy): per-stock forecast losses, the primary statistic D, and the stationary date bootstrap.\n2. Data: parse stored TWSE T86 and MI_INDEX JSON responses into a panel, apply the universe and exclusion rules,\n   and build u (class flows), x (market-reference basis) and the limit indicator.\n3. Run: one period at a time (\"validation\" or \"holdout\"), writing a results JSON.\n\nThe executor never downloads. Downloads are done by the Colab notebook, which stores raw responses with SHA-256.\n\"\"\"\nfrom __future__ import annotations\n\nimport json\nimport math\nimport os\nimport re\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nD_STAR = 0.0035\nSEED = 20261107\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# 1. Core\n# ----------------------------------------------------------------------------------------------------------------\n\ndef _design(u, x, lim, rows, with_x):\n    \"\"\"Rows index the forecast origin t; target is u[t+1].\"\"\"\n    cols = [np.ones(len(rows)), u[rows], u[rows - 1], lim[rows]]\n    if with_x:\n        cols += [x[rows], x[rows - 1]]\n    return np.column_stack(cols)\n\n\ndef stock_losses(u, x, lim, usable, fit_mask, eval_mask):\n    \"\"\"Per-date loss differential for one stock.\n\n    u, x, lim, usable: arrays over the full date axis. usable[t] means day t may serve as a forecast origin\n    (t-1, t, t+1 valid and x_t, x_{t-1} defined). fit_mask / eval_mask select origins for fitting / evaluation.\n    Returns (ell, var_fit) where ell[t] = (e0^2 - e1^2) / var_fit on evaluation origins and NaN elsewhere.\n    \"\"\"\n    T = len(u)\n    t_all = np.arange(1, T - 1)\n    fin = np.isfinite(u)\n    # Amendment A1: an origin needs finite flows at t-1, t, t+1 (a no-op for Foreign; needed for classes that a\n    # T86 schema era does not report).\n    ok = usable[1:T - 1] & fin[0:T - 2] & fin[1:T - 1] & fin[2:T]\n    fit = t_all[ok & fit_mask[1:T - 1]]\n    ev = t_all[ok & eval_mask[1:T - 1]]\n    ell = np.full(T, np.nan)\n    if len(fit) < 50 or len(ev) < 50:\n        return ell, np.nan\n    y_fit = u[fit + 1]\n    var_fit = float(np.var(y_fit))\n    if not np.isfinite(var_fit) or var_fit <= 0:\n        return ell, np.nan\n    b0 = np.linalg.lstsq(_design(u, x, lim, fit, False), y_fit, rcond=None)[0]\n    b1 = np.linalg.lstsq(_design(u, x, lim, fit, True), y_fit, rcond=None)[0]\n    e0 = u[ev + 1] - _design(u, x, lim, ev, False) @ b0\n    e1 = u[ev + 1] - _design(u, x, lim, ev, True) @ b1\n    ell[ev] = (e0 ** 2 - e1 ** 2) / var_fit\n    return ell, var_fit\n\n\ndef panel_D(L):\n    \"\"\"L: stocks x dates, NaN where not an evaluation origin. D = mean over stocks of each stock's mean loss gain.\"\"\"\n    per = np.nanmean(L, axis=1)\n    per = per[np.isfinite(per)]\n    return float(per.mean()), per\n\n\ndef stationary_indices(n, mean_block, rng):\n    p = 1.0 / mean_block\n    idx = np.empty(n, dtype=np.int64)\n    idx[0] = rng.integers(n)\n    for i in range(1, n):\n        idx[i] = rng.integers(n) if rng.random() < p else (idx[i - 1] + 1) % n\n    return idx\n\n\ndef bootstrap_D(L, mean_block=20, B=9999, seed=SEED):\n    \"\"\"Stationary bootstrap over dates, the same resampled dates for every stock.\"\"\"\n    cols = np.where(np.isfinite(L).any(axis=0))[0]\n    Lc = L[:, cols]\n    rng = np.random.default_rng(seed)\n    n = Lc.shape[1]\n    S = np.nan_to_num(Lc)\n    C = np.isfinite(Lc).astype(float)\n    out = np.empty(B)\n    for b in range(B):\n        idx = stationary_indices(n, mean_block, rng)\n        num = S[:, idx].sum(1)\n        cnt = C[:, idx].sum(1)\n        per = num[cnt > 0] / cnt[cnt > 0]\n        out[b] = per.mean()\n    return out\n\n\ndef decide(D, boot):\n    \"\"\"Registered criteria. One-sided p for D > 0 from the bootstrap distribution centred at D.\"\"\"\n    lo, hi = np.percentile(boot, [2.5, 97.5])\n    p = float(np.mean(boot - D >= D)) if D > 0 else 1.0\n    if p < 0.05 and D > 0 and lo > 0:\n        verdict = \"supportive\"\n    elif hi < D_STAR:\n        verdict = \"inconsistent\"\n    else:\n        verdict = \"inconclusive\"\n    return {\"D\": D, \"p_one_sided\": p, \"ci95\": [float(lo), float(hi)], \"D_star\": D_STAR, \"verdict\": verdict}\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# 2. Data\n# ----------------------------------------------------------------------------------------------------------------\n\n# Amendment A1 (7 November 2026, before any statistic was computed). TWSE's T86 report has three schema eras in the\n# registered sample (census of every stored response, schema_census.json):\n#   E1 2013-01-02 .. 2014-11-28: 外資買賣超股數, 投信買賣超股數, 自營商買賣超股數 (no own/hedging split)\n#   E2 2014-12-01 .. 2017-12-15: as E1 plus 自營商買賣超股數(自行買賣) and 自營商買賣超股數(避險)\n#   E3 2017-12-18 ..           : 外陸資買賣超股數(不含外資自營商) and 外資自營商買賣超股數 replace 外資買賣超股數\n# Each series is read from the first name in its list that the day's schema contains. Foreign is the reported foreign\n# total in E1-E2 and the sum of its two parts in E3 (spec: \"Foreign (including foreign dealers)\"). A series that a\n# day's schema does not report is NaN for every stock that day (never zero). An unknown schema stops the run.\nT86_FIELDS = {\n    \"foreign_ex\": [\"外陸資買賣超股數(不含外資自營商)\", \"外資買賣超股數\"],\n    \"foreign_dealer\": [\"外資自營商買賣超股數\", None],     # None: contained in the E1-E2 foreign total, read as 0\n    \"trust\": [\"投信買賣超股數\"],\n    \"dealer_total\": [\"自營商買賣超股數\"],\n    \"dealer_own\": [\"自營商買賣超股數(自行買賣)\"],\n    \"dealer_hedge\": [\"自營商買賣超股數(避險)\"],\n    \"total3\": [\"三大法人買賣超股數\"],\n}\nREQUIRED = (\"foreign_ex\", \"trust\", \"dealer_total\")\nQUOTE_TITLE_SUFFIX = \"權證、牛熊證、可展延牛熊證))\"\nINDEX_NAME = \"發行量加權股價指數\"\n\n\ndef _num(s):\n    s = str(s).replace(\",\", \"\").strip()\n    if s in (\"\", \"--\", \"---\", \"X\"):\n        return math.nan\n    try:\n        return float(s)\n    except ValueError:\n        return math.nan\n\n\ndef t86_positions(fields):\n    \"\"\"Column index per series for one schema, None if the series is contained elsewhere (read as 0), -1 if absent.\"\"\"\n    pos = {}\n    for k, names in T86_FIELDS.items():\n        hit = next((n for n in names if n is not None and n in fields), \"absent\")\n        if hit == \"absent\" and None in names and any(n in fields for n in T86_FIELDS[\"foreign_ex\"][1:]):\n            pos[k] = None\n        else:\n            pos[k] = fields.index(hit) if hit != \"absent\" else -1\n    for k in REQUIRED:\n        if pos[k] is None or pos[k] < 0:\n            raise ValueError(f\"unknown T86 schema: required series {k} not found in {fields}\")\n    return pos\n\n\ndef parse_t86(j):\n    \"\"\"Returns (records, unavailable): records[code][series] and the set of series this day's schema lacks.\"\"\"\n    pos = t86_positions(j[\"fields\"])\n    unavailable = {k for k, i in pos.items() if i is not None and i < 0}\n    out = {}\n    for row in j.get(\"data\", []):\n        code = row[0].strip()\n        if len(code) == 4 and code.isdigit():\n            out[code] = {k: (0.0 if i is None else (math.nan if i < 0 else _num(row[i]))) for k, i in pos.items()}\n    return out, unavailable\n\n\ndef t86_diagnostics(day_files):\n    \"\"\"Flow-only accounting checks per schema era (no prices, no forecasts). Shares of stock-days on which the\n    three-institution total equals the sum of the parts, with and without the foreign dealers, and on which the dealer\n    total equals own + hedging; and the foreign dealers' share of absolute foreign net flow.\"\"\"\n    eras = {}\n    for d, t86, _ in day_files:\n        key = tuple(t86[\"fields\"])\n        e = eras.setdefault(key, {\"first\": d, \"last\": d, \"days\": 0, \"rows\": 0, \"total_with_fd\": 0, \"total_without_fd\": 0,\n                                  \"dealer_rows\": 0, \"dealer_sum_ok\": 0, \"abs_fd\": 0.0, \"abs_foreign\": 0.0})\n        e[\"last\"] = d; e[\"days\"] += 1\n        rec, unavail = parse_t86(t86)\n        for r in rec.values():\n            vals = [r[\"foreign_ex\"], r[\"foreign_dealer\"], r[\"trust\"], r[\"dealer_total\"], r[\"total3\"]]\n            if not all(np.isfinite(v) for v in vals):\n                continue\n            e[\"rows\"] += 1\n            e[\"total_with_fd\"] += abs(r[\"total3\"] - (r[\"foreign_ex\"] + r[\"foreign_dealer\"] + r[\"trust\"] + r[\"dealer_total\"])) < 0.5\n            e[\"total_without_fd\"] += abs(r[\"total3\"] - (r[\"foreign_ex\"] + r[\"trust\"] + r[\"dealer_total\"])) < 0.5\n            if np.isfinite(r[\"dealer_own\"]) and np.isfinite(r[\"dealer_hedge\"]):\n                e[\"dealer_rows\"] += 1\n                e[\"dealer_sum_ok\"] += abs(r[\"dealer_total\"] - r[\"dealer_own\"] - r[\"dealer_hedge\"]) < 0.5\n            e[\"abs_fd\"] += abs(r[\"foreign_dealer\"]); e[\"abs_foreign\"] += abs(r[\"foreign_ex\"] + r[\"foreign_dealer\"])\n    out = []\n    for key, e in eras.items():\n        n = max(e[\"rows\"], 1)\n        out.append({\"first\": e[\"first\"], \"last\": e[\"last\"], \"days\": e[\"days\"], \"rows\": e[\"rows\"],\n                    \"share_total_eq_parts_with_foreign_dealers\": e[\"total_with_fd\"] / n,\n                    \"share_total_eq_parts_without_foreign_dealers\": e[\"total_without_fd\"] / n,\n                    \"share_dealer_total_eq_own_plus_hedge\": (e[\"dealer_sum_ok\"] / e[\"dealer_rows\"]) if e[\"dealer_rows\"] else None,\n                    \"foreign_dealer_share_of_abs_foreign_flow\": e[\"abs_fd\"] / e[\"abs_foreign\"] if e[\"abs_foreign\"] else None,\n                    \"fields\": list(key)})\n    return out\n\n\ndef parse_mi_index(j):\n    quotes, index = {}, math.nan\n    for t in j.get(\"tables\", []):\n        title = t.get(\"title\") or \"\"\n        fields = t.get(\"fields\") or []\n        if title.endswith(QUOTE_TITLE_SUFFIX) and \"證券代號\" in fields:\n            ic, iv, ival = fields.index(\"證券代號\"), fields.index(\"成交股數\"), fields.index(\"成交金額\")\n            icl, isg = fields.index(\"收盤價\"), fields.index(\"漲跌(+/-)\")\n            for row in t.get(\"data\", []):\n                code = row[ic].strip()\n                if len(code) == 4 and code.isdigit():\n                    sign = re.sub(r\"<[^>]+>\", \"\", str(row[isg])).strip()\n                    quotes[code] = {\"vol\": _num(row[iv]), \"value\": _num(row[ival]), \"close\": _num(row[icl]),\n                                    \"event\": sign == \"X\"}\n        if fields and t.get(\"data\") and any(str(r[0]).strip() == INDEX_NAME for r in t[\"data\"]):\n            for r in t[\"data\"]:\n                if str(r[0]).strip() == INDEX_NAME:\n                    index = _num(r[1])\n    return quotes, index\n\n\ndef parse_twt49u(j):\n    \"\"\"Ex-rights / ex-dividend calculation table. Returns a set of (YYYYMMDD, code). Field names are matched by\n    content because probe 2 returned no rows; unparsed tables are reported, not guessed.\"\"\"\n    out = set()\n    f = j.get(\"fields\") or []\n    idate = next((i for i, x in enumerate(f) if \"日期\" in x), None)\n    icode = next((i for i, x in enumerate(f) if \"代號\" in x), None)\n    if idate is None or icode is None:\n        return out\n    for row in j.get(\"data\", []):\n        digits = re.sub(r\"[^0-9]\", \"\", str(row[idate]))\n        if len(digits) == 7:\n            digits = str(int(digits[:3]) + 1911) + digits[3:]\n        code = str(row[icode]).strip()\n        if len(digits) == 8 and len(code) == 4 and code.isdigit():\n            out.add((digits, code))\n    return out\n\n\n@dataclass\nclass Panel:\n    dates: list\n    codes: list\n    vol: np.ndarray       # stocks x dates\n    value: np.ndarray\n    close: np.ndarray\n    event: np.ndarray     # bool\n    flows: dict           # class -> stocks x dates\n    index: np.ndarray     # dates\n\n\ndef build_panel(day_files, extra_events=None):\n    \"\"\"day_files: list of (date 'YYYYMMDD', t86_json, mi_json) in calendar order (trading days only).\n    extra_events: optional set of (YYYYMMDD, code) from TWT49U, united with the X change-sign flag.\"\"\"\n    parsed = []\n    codes = set()\n    for d, t86, mi in day_files:\n        fl, unavail = parse_t86(t86)\n        q, idx = parse_mi_index(mi)\n        parsed.append((d, (fl, unavail), q, idx))\n        codes |= set(q)\n    codes = sorted(codes)\n    ci = {c: i for i, c in enumerate(codes)}\n    N, T = len(codes), len(parsed)\n    nan = lambda: np.full((N, T), np.nan)\n    vol, value, close = nan(), nan(), nan()\n    event = np.zeros((N, T), bool)\n    flows = {k: nan() for k in T86_FIELDS}\n    index = np.full(T, np.nan)\n    for t, (d, (fl, unavail), q, idx) in enumerate(parsed):\n        index[t] = idx\n        for c, rec in q.items():\n            i = ci[c]\n            vol[i, t], value[i, t], close[i, t], event[i, t] = rec[\"vol\"], rec[\"value\"], rec[\"close\"], rec[\"event\"]\n            if rec[\"vol\"] and rec[\"vol\"] > 0:\n                f = fl.get(c)\n                for k in T86_FIELDS:\n                    # absent from the day's report = no institutional trade (0); a series the schema lacks = NaN\n                    flows[k][i, t] = math.nan if k in unavail else (0.0 if f is None else f[k])\n    x_only = int(event.sum())\n    if extra_events:\n        di = {p[0]: t for t, p in enumerate(parsed)}\n        for d, c in extra_events:\n            if d in di and c in ci:\n                event[ci[c], di[d]] = True\n    P = Panel([p[0] for p in parsed], codes, vol, value, close, event, flows, index)\n    P.event_counts = {\"x_flag\": x_only, \"union_with_twt49u\": int(event.sum())}\n    return P\n\n\ndef class_flows(P):\n    F = P.flows\n    out = {\n        \"Foreign\": F[\"foreign_ex\"] + F[\"foreign_dealer\"],\n        \"Trust\": F[\"trust\"],\n        \"DealerOwn\": F[\"dealer_own\"],\n        \"Hedge\": F[\"dealer_hedge\"],\n    }\n    # Residual = minus all reported institutional flow. The dealer total replaces own + hedging, which it equals\n    # wherever the split is reported (checked by t86_diagnostics), and exists in every schema era.\n    out[\"Residual\"] = -(out[\"Foreign\"] + out[\"Trust\"] + F[\"dealer_total\"])\n    return out\n\n\ndef prepare(P, dev_mask, top_n=300, min_share=0.95, trail=60, post_listing=60):\n    \"\"\"Universe, flow scaling, beta, market-reference basis x, limit indicator and usable-origin masks.\"\"\"\n    N, T = P.close.shape\n    traded = np.isfinite(P.vol) & (P.vol > 0) & np.isfinite(P.close)\n    logc = np.log(P.close)\n    ret = np.full((N, T), np.nan)\n    ret[:, 1:] = logc[:, 1:] - logc[:, :-1]\n    prev_close = np.full((N, T), np.nan)\n    prev_close[:, 1:] = P.close[:, :-1]\n    limit = np.abs(P.close - prev_close) >= 0.095 * prev_close\n    limit &= np.isfinite(prev_close)\n    ri = np.full(T, np.nan)\n    ri[1:] = np.diff(np.log(P.index))\n    excl = P.event.copy()\n    excl[:, 1:] |= P.event[:, :-1]\n    # first `post_listing` traded days of a stock that starts trading after the first date\n    first = np.argmax(traded, axis=1)\n    for i in range(N):\n        if first[i] > 0:\n            excl[i, first[i]:first[i] + post_listing] = True\n    # universe on development days\n    dev = dev_mask[None, :]\n    usable_share = (traded & ~excl & dev).sum(1) / max(dev_mask.sum(), 1)\n    turnover = np.where(dev & traded, P.value, np.nan)\n    mean_turn = np.nanmean(turnover, axis=1)\n    elig = np.where((usable_share >= min_share) & np.isfinite(mean_turn))[0]\n    order = elig[np.lexsort((np.array(P.codes)[elig], -mean_turn[elig]))]\n    U = order[:top_n]\n    res = {\"universe_codes\": [P.codes[i] for i in U]}\n    # beta on development days\n    beta = np.full(N, np.nan)\n    for i in U:\n        m = dev_mask & traded[i] & ~excl[i] & ~limit[i] & np.isfinite(ret[i]) & np.isfinite(ri)\n        if m.sum() > 100:\n            X = ri[m]\n            beta[i] = float(np.dot(X - X.mean(), ret[i, m] - ret[i, m].mean()) / np.dot(X - X.mean(), X - X.mean()))\n    # adjusted log level: event-day return replaced by beta * index return; missing returns count as 0\n    adj = np.where(P.event, beta[:, None] * ri[None, :], ret)\n    adj = np.where(np.isfinite(adj), adj, 0.0)\n    level = np.cumsum(adj, axis=1)\n    logI = np.log(P.index)\n    basis = level - beta[:, None] * (logI[None, :] - logI[0])\n    x = np.full((N, T), np.nan)\n    for t in range(trail, T):\n        x[:, t] = basis[:, t] - basis[:, t - trail:t].mean(1)\n    flows = class_flows(P)\n    scale = np.nanmean(np.where(dev & traded, P.vol, np.nan), axis=1)\n    u = {k: v / scale[:, None] for k, v in flows.items()}\n    valid = traded & ~excl\n    usable = np.zeros((N, T), bool)\n    usable[:, 1:-1] = valid[:, :-2] & valid[:, 1:-1] & valid[:, 2:]\n    usable[:, 1:-1] &= np.isfinite(x[:, 1:-1]) & np.isfinite(x[:, :-2])\n    res.update({\"U\": U, \"x\": x, \"u\": u, \"limit\": limit.astype(float), \"usable\": usable, \"beta\": beta})\n    return res\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# 3. Run\n# ----------------------------------------------------------------------------------------------------------------\n\ndef run_period(prep, fit_mask, eval_mask, cls=\"Foreign\", min_share=0.95, B=9999, mean_block=20, seed=SEED):\n    U = prep[\"U\"]\n    rows = []\n    for i in U:\n        if prep[\"usable\"][i][eval_mask].mean() < min_share:\n            continue\n        ell, _ = stock_losses(prep[\"u\"][cls][i], prep[\"x\"][i], prep[\"limit\"][i], prep[\"usable\"][i], fit_mask, eval_mask)\n        if np.isfinite(ell).any():\n            rows.append(ell)\n    L = np.vstack(rows)\n    D, per = panel_D(L)\n    boot = bootstrap_D(L, mean_block=mean_block, B=B, seed=seed)\n    out = decide(D, boot)\n    out.update({\"class\": cls, \"n_stocks\": int(L.shape[0]), \"share_stocks_positive\": float((per > 0).mean()),\n                \"mean_block\": mean_block, \"B\": B})\n    return out\n\n\nif __name__ == \"__main__\":\n    print(\"P1T executor. Use the Colab notebook or validate_executor.py.\")\n"
assert hashlib.sha256(EXECUTOR_SRC.encode()).hexdigest() == EXECUTOR_SHA256, "executor changed"
open("p1t_executor.py", "w").write(EXECUTOR_SRC)
sys.path.insert(0, "."); import p1t_executor as ex

try:
    from google.colab import drive; drive.mount("/content/drive"); ROOT = "/content/drive/MyDrive/P1T_raw"
except Exception: ROOT = "P1T_raw"
os.makedirs(ROOT, exist_ok=True)
for sub in ("T86", "MI", "TWT49U", "FMTQIK"): os.makedirs(os.path.join(ROOT, sub), exist_ok=True)
START, END, DEV_END = "20130102", "20181228", "20161230"
HOLDOUT_START = "20190102"
H = {"User-Agent": "Mozilla/5.0 (academic research; P1T)"}
MAN = os.path.join(ROOT, "manifest.jsonl")
def now(): return datetime.datetime.now(datetime.timezone.utc).isoformat()
def fetch(path, url, params, want_date=None):
    fn = os.path.join(ROOT, path + ".json.gz")
    if os.path.exists(fn):
        return json.loads(gzip.open(fn).read())
    for attempt in range(6):
        try:
            r = requests.get(url, params=params, headers=H, timeout=60)
            if r.status_code == 200:
                j = r.json()
                if want_date is not None and j.get("stat") == "OK" and j.get("date") != want_date:
                    raise SystemExit(f"DATE MISMATCH {path}: asked {want_date}, got {j.get('date')}")
                gzip.open(fn, "wb").write(r.content)
                with open(MAN, "a") as m:
                    m.write(json.dumps({"path": path, "url": r.url, "sha256": hashlib.sha256(r.content).hexdigest(), "utc": now()}) + "\n")
                time.sleep(3.5)
                return j
        except SystemExit: raise
        except Exception as e: print("retry", path, repr(e))
        time.sleep(30 * (attempt + 1))
    raise RuntimeError("failed " + path)

# 1. trading calendar from monthly market summaries
days = []
for y in range(2013, 2019):
    for mth in range(1, 13):
        j = fetch(f"FMTQIK/{y}{mth:02d}", "https://www.twse.com.tw/rwd/zh/afterTrading/FMTQIK",
                  {"date": f"{y}{mth:02d}01", "response": "json"})
        for row in j.get("data", []):
            dg = re.sub(r"[^0-9]", "", row[0]); d = str(int(dg[:-4]) + 1911) + dg[-4:]
            if START <= d <= END: days.append(d)
days = sorted(set(days)); assert days[-1] <= END and all(d < HOLDOUT_START for d in days)
print(len(days), "trading days", days[0], days[-1])

# 2. daily T86 and MI_INDEX
for k, d in enumerate(days):
    fetch(f"T86/{d}", "https://www.twse.com.tw/rwd/zh/fund/T86", {"date": d, "selectType": "ALLBUT0999", "response": "json"}, d)
    fetch(f"MI/{d}", "https://www.twse.com.tw/rwd/zh/afterTrading/MI_INDEX", {"date": d, "type": "ALLBUT0999", "response": "json"}, d)
    if k % 50 == 0: print(k, d, now(), flush=True)

# 3. ex-rights table, month by month
ev = set()
for y in range(2013, 2019):
    for mth in range(1, 13):
        import calendar
        a = f"{y}{mth:02d}01"; b = f"{y}{mth:02d}{calendar.monthrange(y, mth)[1]:02d}"
        j = fetch(f"TWT49U/{y}{mth:02d}", "https://www.twse.com.tw/rwd/zh/exRight/TWT49U", {"startDate": a, "endDate": b, "response": "json"})
        ev |= ex.parse_twt49u(j)

# 4. run the registered executor once on the validation period
def load(sub, d): return json.loads(gzip.open(os.path.join(ROOT, sub, d + ".json.gz")).read())
day_files = [(d, load("T86", d), load("MI", d)) for d in days]
P = ex.build_panel(day_files, ev)   # amendment A1: era-aware T86 parser
dates = np.array(P.dates); dev = dates <= DEV_END; val = ~dev
prep = ex.prepare(P, dev)
res = {"spec": "P1T", "period": "validation", "amendment": "A1", "executor_sha256": EXECUTOR_SHA256, "run_utc": now(),
       "t86_diagnostics": ex.t86_diagnostics(day_files),
       "n_days": {"development": int(dev.sum()), "validation": int(val.sum())}, "event_counts": P.event_counts,
       "twt49u_events_parsed": len(ev), "universe_size": len(prep["universe_codes"]),
       "beta_summary": [float(np.nanpercentile(prep["beta"][prep["U"]], q)) for q in (5, 50, 95)],
       "primary_validation": ex.run_period(prep, dev, val, "Foreign")}
res["robustness_blocks"] = {b: ex.run_period(prep, dev, val, "Foreign", mean_block=b) for b in (10, 60)}
res["secondary_S1_validation"] = {c: ex.run_period(prep, dev, val, c) for c in ("Trust", "DealerOwn")}
res["note"] = "Validation pass only. The registered result is the holdout run; this pass may reveal bugs, which are fixed and logged before the holdout."
json.dump(res, open("p1t_validation_results.json", "w"), indent=1, ensure_ascii=False)
with zipfile.ZipFile("p1t_validation_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("p1t_validation_results.json"); z.write(MAN, "manifest.jsonl")
print(json.dumps(res["primary_validation"], indent=1))
try:
    from google.colab import files; files.download("p1t_validation_results.zip")
except Exception: print("saved p1t_validation_results.zip")
